In [ ]:
# Cell 1 — self-contained bootstrap
from google.colab import userdata
import torch, os, sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_EMAIL = "evenjlinekka@gmail.com"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Evenjlin/deep-space-interference-ml.git"

if not os.path.exists('/content/deep-space-interference-ml'):
    !git clone {REPO_URL} /content/deep-space-interference-ml
%cd /content/deep-space-interference-ml
!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "Evenjlin"
!git remote set-url origin {REPO_URL}
!git pull

!pip install -r requirements.txt -q
sys.path.insert(0, os.getcwd())

In [ ]:
# Cell 2 — denoising-only training data (SOI+noise, NO interference -- BASE-PAPER FACT)
import numpy as np
import torch
from src.channel import SignalConfig, generate_soi, generate_noise
from src.model import MitigationAutoencoder, complex_to_channels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = SignalConfig(fd_max=0.0)
rng = np.random.default_rng(42)
N_SYMBOLS = 64  # must be multiple of 16 for n_stages=2 (4^2 downsample factor)
SNR_DB_TRAIN = 15.0

def make_denoising_pair(cfg, n_symbols, snr_db, rng):
    s, bits, fd, phi_m = generate_soi(cfg, n_symbols, rng)
    w = generate_noise(len(s), rng)
    rho_snr = 10 ** (snr_db / 10)
    x = s + w / np.sqrt(rho_snr)
    return complex_to_channels(x).astype(np.float32), complex_to_channels(s).astype(np.float32)

M_TRAIN, M_VAL = 3000, 750
X_train = np.stack([make_denoising_pair(cfg, N_SYMBOLS, SNR_DB_TRAIN, rng)[0] for _ in range(M_TRAIN)])
Y_train = np.stack([make_denoising_pair(cfg, N_SYMBOLS, SNR_DB_TRAIN, rng)[1] for _ in range(M_TRAIN)])
X_val = np.stack([make_denoising_pair(cfg, N_SYMBOLS, SNR_DB_TRAIN, rng)[0] for _ in range(M_VAL)])
Y_val = np.stack([make_denoising_pair(cfg, N_SYMBOLS, SNR_DB_TRAIN, rng)[1] for _ in range(M_VAL)])
print("X_train:", X_train.shape)

In [ ]:
# Cell 3 — DEBUG LADDER
model = MitigationAutoencoder(n_stages=2, n_hidden=32).to(device)
tiny_x = torch.tensor(X_train[:8], device=device)
out = model(tiny_x)
print("TEST 1: output shape", out.shape, " input shape", tiny_x.shape)
assert out.shape == tiny_x.shape

tiny_y = torch.tensor(Y_train[:8], device=device)
loss_fn = torch.nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=3e-5)  # BASE-PAPER FACT (mitigation AE lr)
loss = loss_fn(out, tiny_y); loss.backward(); opt.step(); opt.zero_grad()
print(f"TEST 2: one grad step OK, loss={loss.item():.4f}")

small_x = torch.tensor(X_train[:200], device=device)
small_y = torch.tensor(Y_train[:200], device=device)
losses = []
for _ in range(20):  # more steps than before -- lr=3e-5 is much smaller, needs more steps to visibly move
    opt.zero_grad()
    out = model(small_x)
    loss = loss_fn(out, small_y)
    loss.backward(); opt.step()
    losses.append(loss.item())
print("TEST 3/4 losses:", [f"{l:.4f}" for l in losses])
assert losses[-1] < losses[0], "Loss did not decrease -- stop, don't proceed"
print("Debug ladder PASSED.")